In [ ]:
"""
Interactive HSV Filter with Fine Hue Control (0.1° Steps)
=========================================================

Unlike standard OpenCV 8-bit HSV (H: 0-179, S/V: 0-255), this tool uses
float32 HSV (H: 0.0-360.0°, S/V: 0.0-1.0) for 20x finer hue resolution.

Hue is controlled via Center + Width (half-range) instead of low/high,
which makes wrap-around at the red boundary (0°/360°) intuitive.

Features:
---------
- Float32 HSV with 0.1° hue precision (3600 steps vs. 180 in 8-bit)
- Automatic hue wrap-around for filtering across 0°/360° boundary
- Live overlay with mask coverage stats (pixel count + percentage)
- Keyboard fine-tuning (0.1° per keystroke)
- Export: binary mask, overlay image, and CSV with all filter settings

Controls:
---------
Trackbars:
  H_center_x10 : Hue center in 0.1° increments (0-3600 -> 0.0-360.0°)
  H_width_x10  : Hue half-width in 0.1° increments (0-1800 -> 0.0-180.0°)
  LS / US       : Saturation min/max (0-255, mapped to 0.0-1.0)
  LV / UV       : Value min/max (0-255, mapped to 0.0-1.0)

Keyboard:
  ESC           : Quit
  SPACE / s     : Save mask, overlay, and CSV
  a / d         : Shift center -0.1° / +0.1°
  w / y         : Increase / decrease width by 0.1°

Hue reference (approximate):
  0°=Red  30°=Orange  60°=Yellow  120°=Green  180°=Cyan  240°=Blue  300°=Magenta
"""

from PIL import Image
import cv2
import numpy as np
import os


def nothing(x):
    pass


# -----------------------------------------------------------
# Window & trackbar setup
# -----------------------------------------------------------
cv2.namedWindow("Tracking")
cv2.resizeWindow("Tracking", 800, 600)

# Hue: center + half-width in 0.1° steps
cv2.createTrackbar("H_center_x10", "Tracking", 0, 3600, nothing)
cv2.createTrackbar("H_width_x10",  "Tracking", 100, 1800, nothing)  # default: 10.0° half-width

# Saturation / Value (0-255, internally scaled to 0-1)
cv2.createTrackbar("LS", "Tracking", 0,   255, nothing)
cv2.createTrackbar("LV", "Tracking", 0,   255, nothing)
cv2.createTrackbar("US", "Tracking", 255, 255, nothing)
cv2.createTrackbar("UV", "Tracking", 255, 255, nothing)

# -----------------------------------------------------------
# Help window
# -----------------------------------------------------------
help_height, help_width = 380, 860
help_img = np.zeros((help_height, help_width, 3), dtype=np.uint8)

lines = [
    "Interactive HSV Filter (float32 precision)",
    "",
    "Trackbars (window 'Tracking'):",
    "  H_center_x10 : Hue center in 0.1 deg (0-3600) -> 0.0-360.0 deg",
    "  H_width_x10  : Hue half-width in 0.1 deg (0-1800) -> 0.0-180.0 deg",
    "  LS / US       : S min / S max (0-255, internally 0-1)",
    "  LV / UV       : V min / V max (0-255, internally 0-1)",
    "",
    "Keyboard (any output window must be focused):",
    "  ESC           : Quit",
    "  SPACE / s     : Save mask, overlay & CSV",
    "  a / d         : Center -0.1 deg / +0.1 deg",
    "  w / y         : Width  +0.1 deg / -0.1 deg",
    "",
    "Output windows:",
    "  Original      : Source image (scaled)",
    "  Mask          : Binary mask (after morphological cleanup)",
    "  Overlay       : Original + red mask overlay + live stats",
]

y0 = 25
dy = 20
for i, line in enumerate(lines):
    cv2.putText(
        help_img, line, (10, y0 + i * dy),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55,
        (200, 200, 200), 1, cv2.LINE_AA,
    )

cv2.namedWindow("Help")
cv2.imshow("Help", help_img)

# -----------------------------------------------------------
# Load image
# -----------------------------------------------------------
image_path = r"C:\path\to\your\image.JPG"

pil_image = Image.open(image_path)
frame = np.array(pil_image)[:, :, ::-1]  # RGB -> BGR

if frame is None:
    print("Failed to load image.")
    raise SystemExit

# Scale to 1080px width, keep aspect ratio
original_height, original_width = frame.shape[:2]
aspect_ratio = original_height / original_width
new_width = 1080
new_height = int(new_width * aspect_ratio)
frame = cv2.resize(frame, (new_width, new_height))

# -----------------------------------------------------------
# Convert to float32 HSV (H: 0-360°, S: 0-1, V: 0-1)
#
# This is the key difference to standard OpenCV 8-bit HSV
# where H is compressed to 0-179 (2° per step).
# Float32 gives full 0.0-360.0° range with arbitrary precision.
# -----------------------------------------------------------
frame_f = frame.astype(np.float32) / 255.0
hsv_f = cv2.cvtColor(frame_f, cv2.COLOR_BGR2HSV)

H = hsv_f[..., 0]  # 0-360 (float)
S = hsv_f[..., 1]  # 0-1
V = hsv_f[..., 2]  # 0-1

kernel = np.ones((3, 3), np.uint8)


# -----------------------------------------------------------
# Helper: draw text block with semi-transparent background
# -----------------------------------------------------------
def draw_text_block(img, lines, origin=(10, 10), font_scale=0.7, thickness=2,
                    font=cv2.FONT_HERSHEY_SIMPLEX, pad=8):
    x, y = origin
    line_h = int(24 * font_scale + 10)

    widths = []
    for ln in lines:
        (w, h), _ = cv2.getTextSize(ln, font, font_scale, thickness)
        widths.append(w)
    block_w = max(widths) + 2 * pad
    block_h = len(lines) * line_h + 2 * pad

    x2 = min(img.shape[1], x + block_w)
    y2 = min(img.shape[0], y + block_h)

    if x >= img.shape[1] or y >= img.shape[0]:
        return img

    roi = img[y:y2, x:x2].copy()
    bg = np.zeros_like(roi, dtype=np.uint8)
    alpha = 0.55
    cv2.addWeighted(bg, alpha, roi, 1 - alpha, 0, roi)
    img[y:y2, x:x2] = roi

    yy = y + pad + line_h - 10
    for ln in lines:
        cv2.putText(img, ln, (x + pad, yy), font, font_scale,
                    (255, 255, 255), thickness, cv2.LINE_AA)
        yy += line_h

    return img


# -----------------------------------------------------------
# Main loop
# -----------------------------------------------------------
while True:
    # Read trackbar values
    H_center_x10 = cv2.getTrackbarPos("H_center_x10", "Tracking")
    H_width_x10 = cv2.getTrackbarPos("H_width_x10", "Tracking")

    l_s = cv2.getTrackbarPos("LS", "Tracking")
    l_v = cv2.getTrackbarPos("LV", "Tracking")
    u_s = cv2.getTrackbarPos("US", "Tracking")
    u_v = cv2.getTrackbarPos("UV", "Tracking")

    # Convert trackbar integers to actual degrees / float ranges
    H_center_deg = H_center_x10 / 10.0
    H_width_deg = H_width_x10 / 10.0

    H_low_deg = (H_center_deg - H_width_deg) % 360.0
    H_high_deg = (H_center_deg + H_width_deg) % 360.0

    l_s_f = l_s / 255.0
    l_v_f = l_v / 255.0
    u_s_f = u_s / 255.0
    u_v_f = u_v / 255.0

    # -------------------------------------------------------
    # Build hue mask with automatic wrap-around
    #
    # Normal case (e.g. 30°-90°): simple range check
    # Wrap case  (e.g. 350°-10°): OR logic across 0°/360° boundary
    # This is essential for filtering red hues cleanly.
    # -------------------------------------------------------
    wrap_active = False
    if H_low_deg <= H_high_deg:
        mask_h = (H >= H_low_deg) & (H <= H_high_deg)
    else:
        wrap_active = True
        mask_h = (H >= H_low_deg) | (H <= H_high_deg)

    # Saturation and value masks
    mask_s = (S >= l_s_f) & (S <= u_s_f)
    mask_v = (V >= l_v_f) & (V <= u_v_f)

    # Combined mask
    mask_bool = mask_h & mask_s & mask_v
    mask = (mask_bool.astype(np.uint8)) * 255

    # Morphological cleanup: remove noise (open) then fill gaps (close)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel, iterations=1)

    # Coverage statistics
    total_px = mask_clean.size
    masked_px = int(np.count_nonzero(mask_clean))
    coverage = (masked_px / total_px) * 100.0 if total_px > 0 else 0.0

    # Red overlay on matched pixels
    overlay = frame.copy()
    overlay[mask_clean > 0] = (0, 0, 255)
    vis = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    # Live info text block
    info_lines = [
        f"H_center = {H_center_deg:6.1f} deg",
        f"H_width  = {H_width_deg:6.1f} deg (half)",
        f"H_low    = {H_low_deg:6.1f} deg",
        f"H_high   = {H_high_deg:6.1f} deg",
        f"Wrap     = {'YES' if wrap_active else 'NO'}",
        f"Coverage = {coverage:6.3f}%  ({masked_px}/{total_px} px)",
        f"S range  = [{l_s:3d}..{u_s:3d}]  (float [{l_s_f:.3f}..{u_s_f:.3f}])",
        f"V range  = [{l_v:3d}..{u_v:3d}]  (float [{l_v_f:.3f}..{u_v_f:.3f}])",
    ]
    vis = draw_text_block(vis, info_lines, origin=(10, 10), font_scale=0.65, thickness=2)

    # Display
    cv2.imshow("Original", frame)
    cv2.imshow("Mask", mask_clean)
    cv2.imshow("Overlay", vis)

    # -------------------------------------------------------
    # Keyboard handling
    # -------------------------------------------------------
    key = cv2.waitKey(1) & 0xFF

    if key == 27:  # ESC
        break

    elif key == ord(' ') or key == ord('s'):
        # Save mask image, overlay image, and filter settings as CSV
        base, ext = os.path.splitext(image_path)
        mask_path = base + "_Mask_floatHSV" + ext
        overlay_path = base + "_Overlay_floatHSV" + ext
        csv_path = base + "_HSV_settings.csv"

        cv2.imwrite(mask_path, mask_clean)
        cv2.imwrite(overlay_path, vis)

        header = [
            "H_center_deg", "H_width_deg",
            "H_low_deg", "H_high_deg",
            "wrap_active",
            "coverage_percent", "masked_px", "total_px",
            "LS", "LV", "US", "UV",
            "LS_f", "LV_f", "US_f", "UV_f",
        ]
        values = [
            f"{H_center_deg:.4f}",
            f"{H_width_deg:.4f}",
            f"{H_low_deg:.4f}",
            f"{H_high_deg:.4f}",
            "1" if wrap_active else "0",
            f"{coverage:.6f}",
            str(masked_px),
            str(total_px),
            str(l_s), str(l_v), str(u_s), str(u_v),
            f"{l_s_f:.6f}",
            f"{l_v_f:.6f}",
            f"{u_s_f:.6f}",
            f"{u_v_f:.6f}",
        ]

        with open(csv_path, "w", encoding="utf-8") as f:
            f.write(",".join(header) + "\n")
            f.write(",".join(values) + "\n")

        print("Saved:")
        print("  ", mask_path)
        print("  ", overlay_path)
        print("  ", csv_path)

    # Fine-tune center: a = -0.1°, d = +0.1°
    elif key == ord('a'):
        H_center_x10 = (H_center_x10 - 1) % 3601
        cv2.setTrackbarPos("H_center_x10", "Tracking", H_center_x10)

    elif key == ord('d'):
        H_center_x10 = (H_center_x10 + 1) % 3601
        cv2.setTrackbarPos("H_center_x10", "Tracking", H_center_x10)

    # Fine-tune width: w = +0.1°, y = -0.1°
    elif key == ord('w'):
        H_width_x10 = min(1800, H_width_x10 + 1)
        cv2.setTrackbarPos("H_width_x10", "Tracking", H_width_x10)

    elif key == ord('y'):  # y for German keyboard layout (where z is on QWERTY)
        H_width_x10 = max(0, H_width_x10 - 1)
        cv2.setTrackbarPos("H_width_x10", "Tracking", H_width_x10)

cv2.destroyAllWindows()